# M02B: Few-Shot Prompting

Zero-shot uses no examples. One-shot uses one. Few-shot uses 3-5 to maximize accuracy.

**What you'll see:**
- Accuracy progression from 0 → 1 → 3 → 5 examples
- Diminishing returns (when more examples stop helping)
- Trade-offs between accuracy and token costs

---

## 🔧 Step 1: Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import openai

load_dotenv(dotenv_path=Path("..") / ".env")

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

MODEL = "gpt-5-mini"


def ask_openai(prompt, model=MODEL):
    """Ask OpenAI a question using the global client."""
    
    try:
        response = client.responses.create(
            model=model,
            input=prompt
        )
        return response.output_text.strip()
        
    except openai.AuthenticationError:
        return "Error: Invalid API key. Check your .env file."
    except openai.RateLimitError:
        return "Error: Rate limit exceeded. Wait and try again."
    except openai.APIConnectionError:
        return "Error: Network issue. Check your internet."
    except openai.BadRequestError:
        return "Error: Bad request. Check model name."
    except Exception as e:
        return f"API Error: {str(e)}"


print("✅ ask_openai() function created!")

---

## 🎯 What is Few-Shot Prompting?

**Few-shot prompting** means providing 3-5 examples to help the AI learn patterns and improve accuracy.

### The Progression
Results vary by task, but generally:

Biggest improvement: 0 → 1 example
Sweet spot: 3-5 examples
Diminishing returns after 5-7 examples

### How it works
Multiple Examples → Model recognizes patterns → Better predictions

### When to use few-shot
- Complex or nuanced tasks
- Need high accuracy
- Have diverse examples showing edge cases

### Trade-offs
- ✅ Much better accuracy and edge case handling
- ⚠️ Uses more tokens (longer prompts)
- ⚠️ Takes longer to craft good examples

---

## 📈 Building Up: The Example Progression

Let's see how accuracy improves as we add examples. We'll classify product reviews by sentiment with increasing numbers of examples.

**Test case:** "Decent quality but overpriced for what you get."

This is a **tricky case** - mixed sentiment, should be classified as `neutral`.

### 🔬 Step 1: Zero-Shot (0 Examples)

Let's start with no examples and see what happens:

In [ ]:
# Zero-shot: No examples
example_review = "Decent quality but overpriced for what you get."

zero_shot_prompt = f"""What is the sentiment of this product review?

Review: "{example_review}"""

print("📊 ZERO-SHOT (0 examples)")
print("="*60)
print(f"Review: {example_review}")
print(f"\nClassification: ", end="")
print(ask_openai(zero_shot_prompt))
print("\n" + "="*60)
print("\n⚠️  May be inconsistent or verbose")

### 🔬 Step 2: One-Shot (1 Example)

Now add ONE example to show the format:

In [ ]:
# One-shot: 1 example
one_shot_prompt = f"""What is the sentiment of product reviews?

Example:
Review: "Amazing product, exceeded expectations!"
positive

Now classify:
Review: "{example_review}"
"""

print("\n📊 ONE-SHOT (1 example)")
print("="*60)
print(f"Review: {example_review}")
print(f"\nClassification: ", end="")
print(ask_openai(one_shot_prompt))
print("\n" + "="*60)
print("\n✅ Better: Consistent format, but may struggle with edge cases")

### 🔬 Step 3: Few-Shot (3 Examples)

Now add 3 examples covering different sentiments:

In [ ]:
# Few-shot: 3 examples
three_shot_prompt = f"""What is the sentiment of product reviews?

Examples:
Review: "Amazing product, exceeded expectations!"
positive

Review: "Terrible quality, broke immediately."
negative

Review: "It works, nothing special."
neutral

Now classify:
Review: "{example_review}"
"""

print("\n📊 FEW-SHOT (3 examples)")
print("="*60)
print(f"Review: {example_review}")
print(f"\nClassification: ", end="")
print(ask_openai(three_shot_prompt))
print("\n" + "="*60)
print("\n✅ Better: Recognizes patterns across categories")

### 🔬 Step 4: Few-Shot (5 Examples)

Add 5 examples including edge cases:

In [ ]:
# Few-shot: 5 examples with edge cases
five_shot_prompt = f"""What is the sentiment of product reviews?

Examples:
Review: "Amazing product, exceeded expectations!"
positive

Review: "Terrible quality, broke immediately."
negative

Review: "It works, nothing special."
neutral

Review: "Good features but too expensive."
neutral

Review: "Fast shipping but product is mediocre."
neutral

Now classify:
Review: "{example_review}"
"""

print("\n📊 FEW-SHOT (5 examples)")
print("="*60)
print(f"Review: {example_review}")
print(f"\nClassification: ", end="")
print(ask_openai(five_shot_prompt))
print("\n" + "="*60)
print("\n✅ Best: Handles edge cases well, recognizes 'mixed' patterns")

### 🔬 Step 5: Many-Shot (10 Examples)

Let's see what happens with 10 examples:

In [ ]:
# Many-shot: 10 examples
ten_shot_prompt = f"""What is the sentiment of product reviews?

Examples:
Review: "Amazing product, exceeded expectations!"
positive

Review: "Best purchase I've made this year!"
positive

Review: "Terrible quality, broke immediately."
negative

Review: "Waste of money, doesn't work."
negative

Review: "It works, nothing special."
neutral

Review: "Average product, does the job."
neutral

Review: "Good features but too expensive."
neutral

Review: "Fast shipping but product is mediocre."
neutral

Review: "Great design, poor execution."
neutral

Review: "Nice idea but overpriced."
neutral

Now classify:
Review: "{example_review}"
"""

print("\n📊 MANY-SHOT (10 examples)")
print("="*60)
print(f"Review: {example_review}")
print(f"\nClassification: ", end="")
print(ask_openai(ten_shot_prompt))
print("\n" + "="*60)
print("\n⚠️  Diminishing returns: Minimal improvement over 5 examples")

---

## 📊 Example Progression Summary

<div style="text-align: left; display: inline-block;">

| Approach | Examples | Typical Results |
|----------|----------|-----------------|
| Zero-shot | 0 | Inconsistent |
| One-shot | 1 | Good consistency |
| Few-shot (3) | 3 | Better accuracy |
| Few-shot (5) | 5 | Excellent results |
| Many-shot (10) | 10 | Minimal improvement |

</div>

### 💡 Key Insights

- **Biggest jump:** 0 → 1 example
- **Sweet spot:** 3-5 examples for most tasks
- **Diminishing returns** after 5-7 examples (minimal gain, higher cost)

---

## 💡 When to Use Each Approach

### 🚀 Zero-Shot (0 examples)
*Best for speed and general knowledge*
- Simple, well-known tasks (translation, summarization)
- Quick prototyping
- General knowledge questions

### 🎯 One-Shot (1 example)
*Best for formatting and style*
- Consistent output formatting
- Tone and style matching
- When you have one perfect example to copy

### 🛠️ Few-Shot (3-5 examples)
*Best for complex reasoning and production*
- Critical accuracy requirements
- Nuanced distinctions between categories
- Edge case handling in production systems

### 🛑 Many-Shot (10+ examples)
*Diminishing returns zone*
- Rarely adds value over 5 examples
- High token costs and increased latency
- Only use if data proves it helps

---

## 🎨 Crafting Effective Few-Shot Examples

**Four principles:**

1. **Diversity** — Cover all categories, include edge cases
2. **Clarity** — Concise examples, consistent formatting
3. **Relevance** — Match your actual use case
4. **Balance** — Don't over-represent one category

### Example Structure:
```
Task description

Example 1: [typical case for category A]
Example 2: [typical case for category B]
Example 3: [typical case for category C]
Example 4: [edge case]
Example 5: [edge case]

Now classify: [your input]
```

---

## 💪 Practice: Build Your Own Few-Shot Prompt

### Step 1: Study This Working Example

Here's a complete email routing prompt with 5 examples:

In [ ]:
# Example: Email department routing with 5 examples
email_routing_prompt = """Route customer emails to the correct department.

Examples:
Email: "I'd like to purchase 50 licenses for my team."
sales

Email: "The app crashes when I try to export data."
technical

Email: "Can you explain the charges on my invoice?"
billing

Email: "I want to cancel my subscription immediately."
billing

Email: "How do I reset my password?"
technical

Now route this email:
Email: "Do you offer enterprise pricing for 100+ users?"
"""

print("Email Routing Example:")
print("=" * 60)
print("Email: 'Do you offer enterprise pricing for 100+ users?'")
print(f"\nRouted to: ", end="")
print(ask_openai(email_routing_prompt))
print("\n" + "=" * 60)

### 💡 Notice
- 5 examples covering main departments with edge cases
- Consistent format throughout
- Clear department labels

---

### Your Turn: Try These Ideas

Use the template below to create your own few-shot prompt.

**Ideas to try:**
- **Code language detection:** Identify programming language from code snippets
- **Urgency classification:** Categorize messages as urgent, normal, or low priority
- **Topic extraction:** Extract main topic from news headlines
- **Tone detection:** Identify tone as professional, casual, or aggressive
- **Product categorization:** Classify products into categories (electronics, clothing, food, etc.)

In [ ]:
# TODO: Replace the [bracketed] text with your own task,
#         examples, and test input. Then uncomment the call (at the bottom).

your_fewshot_prompt = """[Your task description]

Examples:
[Example 1 - typical case category A]
Input: ...
Output: ...

[Example 2 - typical case category B]
Input: ...
Output: ...

[Example 3 - typical case category C]
Input: ...
Output: ...

[Example 4 - edge case]
Input: ...
Output: ...

[Example 5 - edge case]
Input: ...
Output: ...

Now [action]:
Input: [your test input]
Output:"""

# Uncomment to test:
# response = ask_openai(your_fewshot_prompt)
# print(response)

---

## 🎯 Key Takeaways

### What You Learned:

**Few-Shot Prompting:**
- Use 3-5 examples for best results
- Pattern recognition across examples improves accuracy
- Uses more tokens than simpler approaches

**The Progression:**
- 0 examples → inconsistent results
- 1 example → consistent format
- 3-5 examples → strong pattern recognition
- 10+ examples → diminishing returns

**Key Insights:**
1. Sweet spot is 3-5 examples for most tasks
2. Include edge cases in your examples
3. Examples teach patterns, not just format



---

### 📍 Next Step

**M03: Production Prompting** — Structured outputs, error handling, and prompt templates.

---

## 🔧 Troubleshooting

**Not seeing improvement with more examples?**
- Check examples are diverse enough
- Ensure examples match your use case
- Make sure you're showing edge cases

**Results still inconsistent with 5 examples?**
- Task might be too subjective
- Examples might be unclear
- Add more explicit instructions

**Prompts too long?**
- Reduce to 3 examples
- Make examples more concise
- Consider one-shot for simpler cases

**Which examples to include?**
- Start with one per category
- Add edge cases next
- Test and iterate

**API errors?**
- Run Step 1: Setup at the top
- Verify your .env file setup
- Check internet connection

---